# Spatial perturbation benchmark — LOCAL synthetic run
Runs the full 2x2 (seed x propagation) benchmark on **synthetic data** — no server, no real .h5mu, no extra deps (pandas not required). Use this to develop/inspect the pipeline locally.
The server version (`run_benchmark.ipynb`) is identical except its data source is the Saunders `.h5mu` adapter instead of the synthetic generator.

In [ ]:
%matplotlib inline
from spbench.synthetic import make_synthetic
from spbench.config import run_benchmark
from spbench.viz import (plot_2x2, plot_attribution, plot_learned_value,
                         plot_significance_contrast, plot_slope, plot_seed_vs_learned,
                         plot_skill_leaderboard)

## 1. Generate synthetic data (planted seed + propagation effects)

In [ ]:
data = make_synthetic(seed=0)
print('cells', data.n_cells, '| genes', data.n_genes,
      '| perturbations', data.perturbations())

## 2. Run the benchmark (trivial seed + Gaussian baseline + simple GCN)

In [ ]:
PERTS = data.perturbations()
res = run_benchmark(data, perturbations=PERTS, k=15, k_ref=5,
                    gcn_kwargs={'hidden': 32, 'epochs': 20})

## 3. Metrics table (per-perturbation 2x2 + attribution + leakage)
Cells: (1) GT seed + baseline, (2) GT seed + learned, (3) model seed + baseline, (4) model seed + learned (end-to-end). Lower E-distance = better.

In [ ]:
hdr = f"{'pert':6} {'e1':>6} {'e2':>6} {'e3':>6} {'e4':>6} {'seed_cost':>10} {'learned_val':>12} {'leak_ok':>8}"
print(hdr); print('-' * len(hdr))
for p in PERTS:
    g = res['grids'][p]; a = res['attribution'][p]
    print(f"{p:6} {g['1']['energy_prop']:6.3f} {g['2']['energy_prop']:6.3f} "
          f"{g['3']['energy_prop']:6.3f} {g['4']['energy_prop']:6.3f} "
          f"{a['seed_cost']:10.3f} {a['learned_value']:12.3f} {str(res['leakage_pass'][p]):>8}")

## 4. Result figures
Plot the **differences** (not absolute E-distance), aggregated across perturbations and contrasting significant vs non-significant. On synthetic data all three are planted-significant, so here we mark a subset just to demonstrate the figures; on real data pass the MC-spatial significant list.

In [ ]:
SIGNIFICANT = PERTS[:2]   # demo only; on real data use the MC-spatial significant list
plot_skill_leaderboard(res)                    # headline: 0..100% signal captured

In [ ]:
plot_significance_contrast(res, SIGNIFICANT)   # B: the headline contrast

In [ ]:
plot_learned_value(res, SIGNIFICANT)           # A: learned_value per perturbation, sorted

In [ ]:
plot_slope(res, SIGNIFICANT)                   # C: baseline -> learned, consistency + outliers

In [ ]:
plot_seed_vs_learned(res, SIGNIFICANT)         # D: where the error lives

**Single-gene 2x2 (explainer, not a result):**

In [ ]:
plot_2x2(res['grids'][PERTS[0]], title=PERTS[0])

## 5. Ranking (lowest end-to-end E-distance = best)
Note: on synthetic data `learned_value` can be negative (the tiny GCN does not always beat the Gaussian baseline) — that is a legitimate benchmark finding, not a bug.

In [ ]:
print('ranking (best first):', res['ranking'])